# Create MERlin data organization

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/<variant>/<acquisition>/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT027_saving_time/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
MERLIN_DIR   = SAMPLE_DIR / "merlin"
DATA_ORG_DIR = MERLIN_DIR / "dataorganization"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.data_organization import create_data_organization
from MERci.acquisition.dave import annotate_dave_with_round_info, dave_config_filename
from MERci.common.experiment_info import resolve_sample_identity

MICROSCOPE  = "ST2"
# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02-04 used (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")
print(f"DATA_ORG_DIR : {DATA_ORG_DIR}")

SAMPLE_DIR   : C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time
SAMPLE_NAME  : 251225_LT027_saving_time
METADATA_DIR : C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata
SETTINGS_DIR : C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\settings


## Auto-detect frame tables

Lists all `frame-table-*.csv` files in the metadata folder. The bits and cells
frame tables are selected by the kind token in the filename
(`frame-table-bits-*` / `frame-table-cells-*`); transit tables are ignored.
Override `BITS_FT` / `CELLS_FT` manually if needed.

In [2]:
ft_files = sorted(
    METADATA_DIR.glob("frame-table-*.csv"),
    key=lambda p: p.stat().st_mtime, reverse=True
)

print("Frame tables found (newest first):")
for f in ft_files:
    ft = pd.read_csv(f, index_col=0)
    colors = sorted(ft["color"].dropna().unique().astype(int))
    print(f"  {f.name}  →  colors: {colors}")

# Select by the kind token in the filename (frame-table-<kind>-<name>.csv);
# transit frame tables (all-blank) are ignored.
BITS_FT  = next(f for f in ft_files if f.name.startswith("frame-table-bits-"))
CELLS_FT = next(f for f in ft_files if f.name.startswith("frame-table-cells-"))

print(f"\nBits  frame table : {BITS_FT.name}")
print(f"Cells frame table : {CELLS_FT.name}")

Frame tables found (newest first):
  frame-table-transit-blkf2.csv  →  colors: []
  frame-table-bits-blkf15_488f2_560f141_650f141.csv  →  colors: [np.int64(488), np.int64(560), np.int64(650)]
  frame-table-cells-blkf15_405f141_488f143.csv  →  colors: [np.int64(405), np.int64(488)]
  frame-table-focustest-blkf2_488f1.csv  →  colors: [np.int64(488)]
  frame-table-cells-blkf11_405f201_488f203.csv  →  colors: [np.int64(405), np.int64(488)]
  frame-table-bits-blkf21_488f203_560f201_650f201.csv  →  colors: [np.int64(488), np.int64(560), np.int64(650)]

Bits  frame table : frame-table-bits-blkf15_488f2_560f141_650f141.csv
Cells frame table : frame-table-cells-blkf15_405f141_488f143.csv


## Round – bit – color mapping

The round → bit → colour mapping is now defined in **notebook 03** (which derives
`N_HYBS` from it) and saved to `round_bit_color_map.csv`. Here it is read back for the
data organization and the Dave annotation. Re-run notebook 03 to change the mapping.

In [3]:
# round_bit_color is defined & saved in notebook 03; read it back here.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(
        f"{rbc_path} not found — run notebook 03 first "
        f"(it now defines the round–bit–color mapping and N_HYBS)."
    )
rbc_df          = pd.read_csv(rbc_path)
round_bit_color = [(int(r), int(b), int(c))
                   for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
print(f"Loaded {len(round_bit_color)} (round, bit, color) rows from {rbc_path.name}")
print(rbc_df.to_string(index=False))

Loaded 26 (round, bit, color) rows from round_bit_color_map.csv
 round  bit  color
     1    1    647
     1    2    560
     2    3    560
     2    4    647
     3    5    647
     3    6    560
     4    7    647
     4    8    560
     5    9    560
     5   10    647
     6   11    647
     6   12    560
     7   13    647
     7   14    560
     8   15    560
     8   16    647
     9   17    647
     9   18    560
    10   19    647
    10   20    560
    11   21    560
    11   22    647
    12   23    647
    12   24    560
    13   25    647
    13   26    560


## Series patterns from round_info.csv

Reads the bits and cells series patterns so the correct `imageType` and
`imageRegExp` fields are written into the data organization.  
Re-run notebook 03 first if `round_info.csv` is out of date.

In [4]:
round_info_path = METADATA_DIR / "round_info.csv"
round_info      = pd.read_csv(round_info_path)

# Multi-boundary round_info carries per-segment rows (imaging_type in
# {cells, bits, transit} + a 'segment' column). Pick the bits/cells series by
# imaging_type so transit movies are never selected; fall back to name matching
# for the legacy schema.
MULTI_BOUNDARY = "segment" in round_info.columns
if "imaging_type" in round_info.columns:
    itype        = round_info["imaging_type"].astype(str).str.lower()
    bits_rows    = round_info[itype == "bits"]
    cells_rows   = round_info[itype == "cells"]
else:
    bits_rows    = round_info[~round_info["series"].str.contains("cells")]
    cells_rows   = round_info[ round_info["series"].str.contains("cells")]

bits_series  = bits_rows.iloc[0]["series"]
cells_series = cells_rows.iloc[0]["series"]

print(f"Multi-boundary layout : {MULTI_BOUNDARY}")
print(f"Bits  series (sample) : {bits_series}")
print(f"Cells series (sample) : {cells_series}")

if MULTI_BOUNDARY:
    tissues = sorted(round_info["tissue"].dropna().astype(int).unique())
    print(f"\nTissues present       : {tissues}")
    print("NOTE: multi-tissue MERlin analysis is per tissue, and each boundary is a\n"
          "distinct movie (imageType). The cell below builds ONE data-organization from\n"
          "the representative series above; for a per-tissue / per-boundary MERlin run,\n"
          "confirm the intended workflow before relying on this file.")

Multi-boundary layout : False
Bits  series (sample) : hal-st2_01_{fov:04d}
Cells series (sample) : hal-st2-cells_{fov:04d}


## Build and save data organization

In [5]:
readouts  = pd.read_csv(MERCI_DIR / "data" / "readouts.csv")
ft_bits   = pd.read_csv(BITS_FT,  index_col=0)
ft_cells  = pd.read_csv(CELLS_FT, index_col=0)

data_org = create_data_organization(
    bits_frame_table  = ft_bits,
    cells_frame_table = ft_cells,
    round_bit_color   = round_bit_color,
    readouts          = readouts,
    bits_series       = bits_series,
    cells_series      = cells_series,
    include_dapi      = True,
)

out_name = f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
out_path = METADATA_DIR / out_name
data_org.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\n{len(data_org)} rows  ({len(data_org)-1} bits + DAPI)")
data_org

Saved: C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\data_organization_ST2_251225_LT027_saving_time.csv

27 rows  (26 bits + DAPI)


,readoutName,channelName,imageType,imageRegExp,bitNumber,imagingRound,color,frame,zPos,fiducialImageType,fiducialRegExp,fiducialImagingRound,fiducialFrame,fiducialColor
0,b1-RS0015,bit01,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,1,1,647,[],[],hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,1,0,488
1,b2-RS0083,bit02,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,2,1,560,"[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 2...","[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...",hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,1,0,488
2,b3-RS0095,bit03,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,3,2,560,"[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 2...","[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...",hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,2,0,488
3,b4-RS0109,bit04,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,4,2,647,[],[],hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,2,0,488
4,b5-RS0175,bit05,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,5,3,647,[],[],hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,3,0,488
5,b6-RS0237,bit06,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,6,3,560,"[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 2...","[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...",hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,3,0,488
6,b7-RS0247,bit07,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,7,4,647,[],[],hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,4,0,488
7,b8-RS0255,bit08,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,8,4,560,"[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 2...","[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...",hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,4,0,488
8,b9-RS0307,bit09,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,9,5,560,"[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 2...","[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...",hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,5,0,488
9,b10-RS0332,bit10,hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,10,5,647,[],[],hal-st2,(?P<imageType>[\w|-]+)_(?P<imagingRound>[\w|-]...,5,0,488


## Annotate Dave XML with bit information

Adds per-round XML comments to the Dave config generated by notebook 04.
Imaging round 1 is the cells acquisition (no bits), so the bit comments attach to
the fluidics loops that precede each bits imaging round (rounds 2…N+1). The
hyb/bit-indexed `round_bit_color` is shifted `+1` here to match those imaging-round
numbers.

Re-run this cell whenever the round–bit–color mapping changes.

In [6]:
N_HYBS    = int(rbc_df["round"].max())
dave_path = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

if not dave_path.exists():
    print(f"No {dave_path.name} found in settings/ — run notebook 04 first.")
else:
    print(f"Annotating: {dave_path.name}")
    # round_bit_color is hyb/bit-indexed (1..N) for data-organization, but the
    # Dave recipe images bits in imaging rounds 2..N+1 (round 1 = cells), so the
    # annotation round indices are shifted +1 to line up with the Fluidics loops.
    annotate_rbc = [(r + 1, bit, color) for (r, bit, color) in round_bit_color]
    annotate_dave_with_round_info(dave_path, annotate_rbc)
    print("Done. Preview of annotated file:")
    with open(dave_path, encoding="ISO-8859-1") as fh:
        print(fh.read())

Annotating: dave-st2-13hybs-251225_LT027_saving_time.xml
Done. Preview of annotated file:
<?xml version="1.0" encoding="ISO-8859-1"?>
<recipe>
    <command_sequence>

        <!-- Hyb 01:
                Bit 1 (647 nm)
                Bit 2 (560 nm)
        -->
        <loop name="Hyb 01 Fluidics">
            <variable_entry name="Hyb 01 Fluidics"/>
        </loop>
        <change_directory>C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\hybs\H01</change_directory>
        <loop name="Hyb 01 Imaging">
            <movie>
                <name increment="Yes">hal-st2_01</name>
                <length>299</length>
                <parameters>hal-config-st2-bits-blkf15_488f2_560f141_650f141</parameters>
                <check_focus>
                    <num_focus_checks>50</num_focus_checks>
                    <focus_scan/>
                </check_focus>
                <overwrite>False</overwrite>
                <variable_entry name="Hyb 01 Ima